# 04 — Session d'entraînement (Colab T4)

**Objectif** : lancer une **vraie** session d'entraînement sur GPU, robuste aux coupures Colab grâce à la reprise automatique sur checkpoint.

**Robustesse aux coupures**

- Sauvegarde automatique de `last.pt` **toutes les 15 min** (et pas seulement à la fin de chaque epoch). Au pire on perd 15 min de calcul.
- Log de `train_loss_step` à wandb **toutes les 50 batches** → la courbe d'apprentissage apparaît dès les premières minutes, sans attendre la fin d'epoch.
- **Si Colab coupe** : relancer toutes les cellules depuis le haut. La cellule de reprise détectera `last.pt` sur Drive et continuera là où ça s'est arrêté.

## 0. Anti-idle Colab (optionnel mais recommandé)

Colab Free coupe les sessions inactives au bout de ~90 min. Pour éviter ça pendant un long entraînement :

1. **Garde cet onglet visible** (au moins au premier plan, pas minimisé).
2. **Optionnel — snippet JS anti-idle** : ouvre la console développeur du navigateur (`F12`), va dans l'onglet *Console*, colle ce code et appuie sur Entrée :

```js
// Clique sur le bouton "Connect" toutes les 60s pour signaler de l'activité.
setInterval(() => {
  const btn = document.querySelector('colab-toolbar-button#connect');
  if (btn) btn.click();
  console.log('[anti-idle] tick', new Date().toLocaleTimeString());
}, 60_000);
```

3. **Ne mets pas le PC en veille** (Paramètres Windows → Alimentation → « Jamais »). L'écran peut s'éteindre, c'est ok.

Ces protections sont en plus du checkpoint toutes les 15 min — même si Colab finit par couper, tu reprends sans perdre plus d'un quart d'heure.

## 1. Setup Colab (Drive + repo + dépendances)

À ignorer si on tourne en local. Sinon, exécuter dans l'ordre.

In [ ]:
# Montage Drive (Colab uniquement)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab :", IN_COLAB)

In [ ]:
# Clone / pull du repo + cd dedans + install
import os, subprocess
REPO_DIR = "/content/Filtre-Voix-DL" if IN_COLAB else os.path.abspath("..")
REPO_URL = "https://github.com/Theo-Lempereur/Filtre-Voix-DL.git"
BRANCH   = "features/training"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR :", REPO_DIR)

In [ ]:
# Vérification GPU
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## 2. Wandb (optionnel)

Si on veut un dashboard en ligne. Sinon, on met `use_wandb=False` plus bas — les logs JSONL locaux suffisent.

In [ ]:
# Décommenter et exécuter une seule fois pour s'authentifier :
# import wandb; wandb.login()

## 3. Configuration du run

Le `run_id` doit être **unique** : il sert de nom de dossier pour les checkpoints (`checkpoints/<run_id>/`) et les logs (`logs/<run_id>/`).

**Convention** : `YYYYMMDD-<tag>`. ⚠️ Si tu reprends un run lancé un jour précédent, **mets le `RUN_ID` en dur** (ex : `RUN_ID = "20260524-base32-lr1e3"`) pour que la reprise détecte bien le bon dossier.

In [ ]:
from src import config as cfg
from datetime import datetime

RUN_TAG = "base32-lr1e3"             # <-- éditer pour chaque expérience
RUN_ID  = f"{datetime.now():%Y%m%d}-{RUN_TAG}"
# Pour reprendre un run d'hier, décommenter et éditer :
# RUN_ID = "20260524-base32-lr1e3"

run_config = {
    "run_id":                    RUN_ID,
    "lr":                        1e-3,
    "weight_decay":              0.0,
    "batch_size":                8,
    "num_workers":               2,
    "num_epochs":                50,
    "base_channels":             32,
    "seed":                      42,
    "grad_clip_norm":            1.0,
    "early_stop_patience":       7,
    "lr_patience":               3,
    "keep_last_n_ckpt":          3,
    "ckpt_every_n_epochs":       1,
    # --- Anti-coupure Colab ---
    "intra_epoch_save_seconds":  15 * 60,   # save last.pt toutes les 15 min
    "intra_epoch_log_every":     50,        # log train_loss_step toutes les 50 batches
    # --- Wandb ---
    "use_wandb":                 True,      # mettre à False si pas de compte wandb
    "notes":                     "Baseline U-Net base=32, MSE magnitude.",
}
print("RUN_ID :", RUN_ID)

## 4. Reprise automatique sur checkpoint

Si un `last.pt` existe déjà dans `checkpoints/<run_id>/`, on le détecte et on le passe à `train(resume_from=...)`. Sinon, on démarre à zéro.

**C'est cette cellule qui rend la session robuste aux coupures Colab.** Avec la sauvegarde intra-epoch toutes les 15 min, on perd au pire un quart d'heure de calcul.

In [ ]:
from pathlib import Path
from src.checkpoint import find_latest_checkpoint

ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
resume_from = find_latest_checkpoint(ckpt_dir)

if resume_from is None:
    print(f"[reprise] aucun checkpoint dans {ckpt_dir} → entraînement neuf.")
    # Au cas où on aurait oublié de mettre le bon RUN_ID, on liste les autres
    # runs présents sur Drive pour aider à diagnostiquer.
    ckpt_root = Path(cfg.CHECKPOINTS)
    if ckpt_root.exists():
        others = sorted(p.name for p in ckpt_root.iterdir() if p.is_dir())
        if others:
            print("[reprise] autres runs présents sur Drive (si tu voulais reprendre l'un d'eux, mets son nom dans RUN_ID) :")
            for name in others:
                print(f"           - {name}")
else:
    print(f"[reprise] checkpoint détecté : {resume_from}")
    print("          l'entraînement reprendra à partir de cet état.")

## 5. Lancement de l'entraînement

**Première fois ?** Lance d'abord un mini-test pour vérifier que tout marche, en décommentant la version avec `max_train_samples=500` (≈ 5 min sur T4).

In [ ]:
from src.train import train

# --- Mini-test pour valider le pipeline (recommandé la première fois) ---
# history = train(run_config, resume_from=resume_from,
#                 max_train_samples=500, max_val_samples=100)

# --- Vrai entraînement sur tout le dataset ---
history = train(run_config, resume_from=resume_from)

## 6. Vérification — les checkpoints sont bien sur Drive

Tolérant : si rien n'a été sauvegardé (entraînement interrompu très tôt), affiche une explication au lieu de planter.

In [ ]:
ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
log_path = Path(cfg.LOGS) / RUN_ID / "history.jsonl"

print(f"Dossier checkpoints : {ckpt_dir}")
if not ckpt_dir.exists():
    print("  ! ce dossier n'existe pas — aucune sauvegarde n'a été faite.")
    print("    Cause probable : entraînement interrompu avant la première")
    print("    sauvegarde (15 min ou fin d'epoch).")
else:
    files = sorted(ckpt_dir.iterdir())
    if not files:
        print("  ! dossier vide.")
    else:
        for p in files:
            size_mb = p.stat().st_size / 1e6
            print(f"  {p.name:<20s}  {size_mb:6.1f} MB")

print(f"\nJSONL d'historique : {log_path}")
print("  existe :", log_path.is_file())

## 7. Aperçu rapide

Quatre courbes pour avoir une idée immédiate. Pour l'analyse fine (gap, flags d'overfit, comparaison de runs) → notebook [05_training_analysis.ipynb](05_training_analysis.ipynb).

In [ ]:
import matplotlib.pyplot as plt

# Protection : si l'entraînement a été interrompu très tôt, history peut être vide
# OU la variable peut ne pas exister du tout (cellule 5 non exécutée après un
# restart). On essaie de la reconstruire depuis le JSONL dans ce cas.
try:
    history
except NameError:
    from src.logging_utils import read_history, metrics_records
    print("[aperçu] history absente du kernel → lecture depuis le JSONL")
    recs = [r for r in metrics_records(read_history(Path(cfg.LOGS) / RUN_ID))
            if not r.get("is_step")]   # on garde uniquement les points par epoch
    history = {k: [r.get(k) for r in recs]
               for k in ["epoch", "train_loss", "val_loss", "val_si_sdr", "lr", "gap"]}

if not history.get("epoch"):
    print("[aperçu] aucune donnée d'epoch à afficher — l'entraînement a été")
    print("          interrompu avant la fin de la première epoch.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    ep = history["epoch"]

    axes[0, 0].plot(ep, history["train_loss"], label="train")
    axes[0, 0].plot(ep, history["val_loss"],   label="val")
    axes[0, 0].set_title("Loss MSE"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(ep, history["val_si_sdr"], color="tab:green")
    axes[0, 1].set_title("SI-SDR validation (dB, plus haut = mieux)"); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(ep, history["lr"], color="tab:orange")
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Learning rate"); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(ep, history["gap"], color="tab:red")
    axes[1, 1].set_title("Gap (val - train) — un gap qui augmente = overfit")
    axes[1, 1].axhline(0, color="k", linestyle="--", linewidth=0.5)
    axes[1, 1].grid(alpha=0.3)

    for ax in axes.flatten():
        ax.set_xlabel("epoch")
    fig.tight_layout()
    plt.show()